# Feature Importance Workflow (Steps 3a–3b)

Run this **after** cohorts exist (Step 2; see `1_cohort_workflow.ipynb`). One cell per cohort for Step 3a and for Step 3b; run Configuration and Sync once, then the cohort cell(s) you need.

## Order of operations

1. **Configuration** — Project root, data root, cohort/age-band list (run once).
2. **Sync inputs** — Sync `gold/cohorts` from S3 to local/NVMe (idempotent).
3. **Step 3a** — MC-CV feature importance per cohort: Cohort 1 (OPIOID_ED), Cohort 2 (POLYPHARMACY). Each cell runs that cohort for all its age_bands; checkpoint skip per cohort/age_band.
4. **Step 3b** — Feature Importance EDA (interactive): one runnable cell **per cohort**. Run the cell for the cohort you want; each runs that cohort for all its age_bands. Checkpoint skip per cohort/age_band.

## Cohorts

| Cohort | Age bands |
|--------|-----------|
| **OPIOID_ED** | 13-24, 25-44, 45-54, 55-64 |
| **POLYPHARMACY** (non_opioid_ed) | 65-74, 75-84, 85-94 |

## Reference

- Step 3a: `3a_feature_importance/run_mc_feature_importance.py`
- Step 3b: `3b_feature_importance_eda/feature_importance_eda_workflow.py`; interactive EDA in `3b_feature_importance_eda/`
- Step 6: `6_final_model/` — model training and selection.

## Configuration

In [ ]:
import sys
import os
from pathlib import Path
import subprocess
import logging

PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path.cwd()
if not (PROJECT_ROOT / "3a_feature_importance").exists():
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from py_helpers.env_utils import get_data_root
from py_helpers.workflow_sync_checkpoint import (
    sync_s3_to_local,
    check_step_checkpoint_exists,
    save_step_checkpoint,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

PYTHON_BIN = Path(sys.executable)
S3_BUCKET = os.environ.get("PGX_S3_BUCKET", "pgxdatalake")
DATA_ROOT = get_data_root()
AWS_PROFILE = os.environ.get("AWS_PROFILE")

COHORTS = {
    "opioid_ed": ["13-24", "25-44", "45-54", "55-64"],
    "non_opioid_ed": ["65-74", "75-84", "85-94"],
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Data root (NVMe/local): {DATA_ROOT}")
print(f"Python: {PYTHON_BIN}")

## Sync required inputs from S3 to NVMe (idempotent)

Sync **gold/cohorts** from S3 so Step 3a can read cohort parquet from local/NVMe. **Idempotent:** `aws s3 sync` only updates changed or missing files.

In [ ]:
# Sync gold/cohorts from S3 to local/NVMe (required for 3a feature importance)
s3_cohorts = f"s3://{S3_BUCKET}/gold/cohorts/"
local_cohorts = DATA_ROOT / "gold" / "cohorts"
ok = sync_s3_to_local(s3_cohorts, local_cohorts, profile=AWS_PROFILE)
print(f"  gold/cohorts: {'OK' if ok else 'FAILED or skipped (no AWS CLI)'}")

## Step 3a: MC-CV feature importance

Monte Carlo CV feature importance (CatBoost, XGBoost, XGBoost RF). One cell per cohort; each runs that cohort for all its age_bands. Skipped when checkpoint exists for that cohort/age_band.

### Cohort 1: OPIOID_ED

In [ ]:
# Step 3a for OPIOID_ED (all age_bands); skip if checkpoint exists per cohort/age_band
step_name_3a = "3a_feature_importance"
script_3a = PROJECT_ROOT / "3a_feature_importance" / "run_mc_feature_importance.py"
cohort = "opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
        print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3a for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

### Cohort 2: POLYPHARMACY (non_opioid_ed)

In [ ]:
# Step 3a for POLYPHARMACY (all age_bands); skip if checkpoint exists per cohort/age_band
cohort = "non_opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3a, cohort, age_band, logger):
        print(f"Step 3a already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3a for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3a), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3a, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")

## Step 3b: Feature Importance EDA (interactive)

BupaR post-target analysis and refined `cohort_feature_importance.csv` in `3b_feature_importance_eda/outputs/`. **One notebook cell per cohort** so you can run interactively: run the cell for the cohort you want. Each cell runs that cohort for all its age_bands; skipped when checkpoint exists for that cohort/age_band.

### Cohort 1: OPIOID_ED

In [ ]:
# Step 3b for OPIOID_ED (all age_bands); skip if checkpoint exists per cohort/age_band
STEP3B_DIR = PROJECT_ROOT / "3b_feature_importance_eda"
script_3b = STEP3B_DIR / "feature_importance_eda_workflow.py"
step_name_3b = "3b_feature_importance_eda"
cohort = "opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
        print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")
print(f"Step 3b outputs: {STEP3B_DIR / 'outputs'}")

### Cohort 2: POLYPHARMACY (non_opioid_ed)

In [ ]:
# Step 3b for POLYPHARMACY (all age_bands); skip if checkpoint exists per cohort/age_band
cohort = "non_opioid_ed"
for age_band in COHORTS[cohort]:
    if check_step_checkpoint_exists(step_name_3b, cohort, age_band, logger):
        print(f"Step 3b already completed for {cohort}/{age_band}. Skipping.")
        continue
    print(f"Running Step 3b for {cohort}/{age_band}...")
    result = subprocess.run(
        [str(PYTHON_BIN), str(script_3b), "--cohort", cohort, "--age-band", age_band],
        cwd=str(PROJECT_ROOT),
    )
    if result.returncode == 0:
        save_step_checkpoint(step_name_3b, cohort, age_band, logger=logger)
    print(f"  {cohort}/{age_band} exit code: {result.returncode}")